# Fracture Classifier — Kaggle Training Notebook

Runs the full pipeline (split → train → evaluate) on Kaggle GPU.

## Setup (once)
1. Upload your image folder as a **Kaggle Dataset** — note the dataset slug
2. Upload the code files (`config.py`, `dataset.py`, `dataloader.py`, `model.py`, `train.py`, `evaluate.py`) as a second **Kaggle Dataset** — note that slug too
3. Enable **GPU accelerator** in notebook settings (T4 or P100/A100)
4. Fill in the two slugs in Cell 2

## Model options
| Model | IMG_SIZE | BATCH_SIZE | VRAM | ~min/epoch |
|---|---|---|---|---|
| `efficientnet_b0` | 224 | 32 | ~4 GB | ~1–2 min |
| `tf_efficientnetv2_m` | 480 | 16 | ~10 GB | ~4–6 min |
| `tf_efficientnetv2_m` | 384 | 24 | ~8 GB | ~3–4 min |
| `convnext_base` | 384 | 16 | ~8 GB | ~3–4 min |

In [ ]:
# Cell 1 — Install extra packages (timm and optuna not preinstalled on Kaggle)
!pip install timm optuna -q

In [ ]:
import os
from pathlib import Path

# ══════════════════════════════════════════════════════════════════════════════
# EDIT THESE TWO LINES — replace with your actual Kaggle dataset slugs
# ══════════════════════════════════════════════════════════════════════════════
IMAGE_DATASET_SLUG = "venthanforearm-xrays"         # slug of the X-ray image dataset
CODE_DATASET_SLUG  = "venthanfracture-pipeline"     # slug of the uploaded code dataset
# ══════════════════════════════════════════════════════════════════════════════

# ── Auto-detect DATA_ROOT (handles both /kaggle/input/{slug}/ and
#    /kaggle/input/datasets/{username}/{slug}/ path structures) ────────────
_img_candidates = [Path(f"/kaggle/input/{IMAGE_DATASET_SLUG}/alle Bilder")]
_datasets_dir = Path("/kaggle/input/datasets")
if _datasets_dir.exists():
    _img_candidates += [
        p / "alle Bilder"
        for p in _datasets_dir.glob(f"*/{IMAGE_DATASET_SLUG}")
    ]

_data_root = next((p for p in _img_candidates if p.exists()), None)

if _data_root is None:
    # Last resort: find 'alle Bilder' anywhere under /kaggle/input
    hits = list(Path("/kaggle/input").rglob("alle Bilder"))
    if hits:
        _data_root = hits[0]

if _data_root is None:
    raise FileNotFoundError(
        f"'alle Bilder' folder not found under /kaggle/input/\n"
        f"Available inputs: {os.listdir('/kaggle/input/')}\n"
        f"IMAGE_DATASET_SLUG = '{IMAGE_DATASET_SLUG}'\n"
        f"Tip: Add the image dataset via 'Add data' in notebook settings."
    )

os.environ["DATA_ROOT"]      = str(_data_root)
os.environ["SPLITS_CSV"]     = "/kaggle/working/data/splits.csv"
os.environ["CHECKPOINT_DIR"] = "/kaggle/working/checkpoints"
os.environ["LOG_DIR"]        = "/kaggle/working/logs"

# ── Model (choose one row from the table in Cell 0) ────────────────────────
os.environ["MODEL_NAME"]  = "tf_efficientnetv2_m"
os.environ["IMG_SIZE"]    = "480"
os.environ["BATCH_SIZE"]  = "16"
os.environ["NUM_WORKERS"] = "4"

# ── Optional tuning knobs (uncomment to override config.py defaults) ────────
# os.environ["HEAD_DROPOUT"]    = "0.4"
# os.environ["CROP_SCALE_MIN"]  = "0.6"

# ── Verify GPU ─────────────────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), "No GPU found — enable GPU in notebook settings!"
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"DATA_ROOT  : {os.environ['DATA_ROOT']}")
print(f"Config: MODEL={os.environ['MODEL_NAME']}  IMG={os.environ['IMG_SIZE']}px  BATCH={os.environ['BATCH_SIZE']}")

In [ ]:
# Cell 3 — Add code dataset to sys.path (handles both Kaggle path structures)
import sys
from pathlib import Path

# Kaggle mounts datasets at two possible locations:
#   Standard:  /kaggle/input/{CODE_DATASET_SLUG}/
#   Newer:     /kaggle/input/datasets/{username}/{CODE_DATASET_SLUG}/
_candidates = [Path(f"/kaggle/input/{CODE_DATASET_SLUG}")]
_datasets_dir = Path("/kaggle/input/datasets")
if _datasets_dir.exists():
    _candidates += list(_datasets_dir.glob(f"*/{CODE_DATASET_SLUG}"))

code_path = next((p for p in _candidates if (p / "dataset.py").exists()), None)

# Last resort: recursive search across all of /kaggle/input
if code_path is None:
    hits = list(Path("/kaggle/input").rglob("dataset.py"))
    if hits:
        code_path = hits[0].parent

if code_path is None:
    import os
    raise FileNotFoundError(
        f"dataset.py not found under /kaggle/input/\n"
        f"Available inputs: {os.listdir('/kaggle/input/')}\n"
        f"CODE_DATASET_SLUG = '{CODE_DATASET_SLUG}'\n"
        f"Tip: Add the code dataset via 'Add data' in notebook settings."
    )

sys.path.insert(0, str(code_path))
print(f"Code path: {code_path}")

In [ ]:
# Cell 4 — Create output directories and build train/val/test split CSV
from pathlib import Path
Path("/kaggle/working/data").mkdir(parents=True, exist_ok=True)
Path("/kaggle/working/checkpoints").mkdir(parents=True, exist_ok=True)
Path("/kaggle/working/logs").mkdir(parents=True, exist_ok=True)

import dataset
dataset.main()

In [ ]:
# Cell 5 — Train (Phase 1: frozen backbone → Phase 2: fine-tune last 2 blocks)
# Expected runtime on T4 at 480px:
#   Phase 1 (20 epochs × ~5 min) ≈ 100 min
#   Phase 2 (up to 40 epochs × ~5 min, early stop) ≈ 60–200 min
#   Total ≈ 2.5–5 hours  (well within Kaggle's 9-hour session limit)
import train
train.main()

In [ ]:
# Cell 6 — Evaluate on the held-out test set
import evaluate
evaluate.main()

In [ ]:
# Cell 7 — Display result plots inline
from IPython.display import Image, display
from pathlib import Path

log_dir = Path("/kaggle/working/logs")
for plot in ["roc_curve.png", "confusion_matrix.png"]:
    path = log_dir / plot
    if path.exists():
        print(f"\n{plot}:")
        display(Image(filename=str(path)))